In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin

from tqdm import tqdm

from sklearn.pipeline import Pipeline

from sklearn.ensemble import GradientBoostingRegressor




from skorch import NeuralNetRegressor

ModuleNotFoundError: No module named 'torch'

In [ ]:
df_final = pd.read_excel("data/processed/df_final.xlsx")
df_final = df_final.sort_values("Date").reset_index(drop=True) 

In [ ]:
covariates = ["dolvol_lag2", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "cashpr", "chinv", "chsh", "depr", "dy", "ep", "invest", "rd_mve", "sp", "nincr"]

In [ ]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=192):
    """
    Génère des splits temporels:
    - Train cumulatif (augmente d'un an à chaque refit)
    - Train 306 mois (= 85% de la data)
    - Validation = fenêtre fixe glissante de 1 an
    - Test = 1 an 
    - Avance de step_months à chaque itération : 1 an

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop quand on a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Décale fenêtre de un → on réactualise tous les 1 ans
        start += step_months

    return splits

In [ ]:
def preprocess_split(x_train, x_val, x_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).
    """
    means_by_ticker = x_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for c in covariates:
            m = means_by_ticker[c].to_dict()
            df[c] = df[c].fillna(df["Ticker"].map(m))
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    assert not x_train_imp[covariates].isna().any().any()
    assert not x_val_imp[covariates].isna().any().any()
    assert not x_test_imp[covariates].isna().any().any()
    
    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

In [ ]:
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target])
    y = subset[target]
    return x, y

In [ ]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final)

preprocessed_splits = []
first_date_split = []

for train_idx, val_idx, test_idx in tqdm(splits):
    first_date_split.append(df_final.loc[test_idx, "Date"].iloc[0])
    x_train, y_train = get_x_y(df_final, train_idx)
    x_val, y_val = get_x_y(df_final, val_idx)
    x_test, y_test = get_x_y(df_final, test_idx)

    #Imputation + Normalisation
    x_train, x_val, x_test = preprocess_split(x_train, x_val, x_test, covariates) #on enlève ticker et date

    preprocessed_splits.append((x_train, y_train, x_val, y_val, x_test, y_test))

In [ ]:
"""Fonctions pour nos métriques : 
def r2: mesure le r2 selon la définition de Gu et al 
% ratio : success ratio, semblable à ce qui est fait dans le papier de Xiu et Liu
R2 benchmark : on compare le R2 de nos modèles à l'historical average 
"""

#r2
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#R2 benchmark 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  
    return (sign_true == sign_pred).mean()

In [ ]:
#NEURAL NETWORK N1
# Réseau de neurones simple avec 1 seule couche cachée

class Net2(nn.Module):
    def __init__(self, n_feature, n_hidden1=64, n_hidden2=32, n_output=1, dropout=0.0):
        super(Net2, self).__init__()
        self.hidden1  = nn.Linear(n_feature, n_hidden1)
        self.dropout1 = nn.Dropout(dropout)
        self.hidden2  = nn.Linear(n_hidden1, n_hidden2)
        self.dropout2 = nn.Dropout(dropout)
        self.output   = nn.Linear(n_hidden2, n_output)

    def forward(self, x):
        x = self.hidden1(x)
        x = self.dropout1(x)
        x = F.relu(x)
        x = self.hidden2(x)
        x = self.dropout2(x)
        x = F.relu(x)
        x = self.output(x)
        return x
    
nn2_architecture = {
    "module__n_feature": x_train.shape[1],  #nb de covariates
    "module__n_hidden1": 64,
    "module__n_hidden2": 32,
    "module__n_output": 1,
}

# Grille d’hyperparamètres
nn2_param = {
    "module__dropout": [0.0, 0.1, 0.3],
    "lr": [0.001, 0.01],
    "optimizer__weight_decay": [0.1, 0.01, 0.001]
}

r2_in_sample_list_nn2 = []
r2_test_list_nn2      = []
best_param_nn2        = []
mse_val_grids_nn2     = []

all_dates_nn2 = []
all_y_true_nn2 = []
all_y_pred_nn2 = []


for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test, dates_test) in enumerate(tqdm(preprocessed_splits), start=1):
    
    #pytorch demande des objets particuliers : des float32 (de base on est en float64) et des ndarray (de base on est en dataframe). aussi y est en 1D, comme x est en 2D pytoch s'attend à du 2D
    y_train_nn = y_train.values.reshape(-1,1).astype(np.float32)
    y_val_nn   = y_val.values.reshape(-1,1).astype(np.float32)
    y_test_nn  = y_test.values.reshape(-1,1).astype(np.float32)

    x_train_nn = x_train.values.astype(np.float32)
    x_val_nn   = x_val.values.astype(np.float32)
    x_test_nn  = x_test.values.astype(np.float32)

    best_mse = float('inf')
    mse_grid = []

    for param in ParameterGrid(nn1_param):
        #epoch: le réseau voit toutes les données d’entraînement une fois. max_epochs=500 = le réseau passera 500 fois sur tout le jeu d’entraînement
        nn1 = NeuralNetRegressor(Net1, verbose=0, max_epochs=500, optimizer=torch.optim.SGD,
        **nn1_architecture, **param) #**nn1_architecture = déplie le dictionnaire pour récupérer directement

        nn1.fit(x_train_nn, y_train_nn)
        y_val_pred = nn1.predict(x_val_nn)
        mse = mean_squared_error(y_val_nn, y_val_pred)
        mse_grid.append((param, mse))

        if mse < best_mse:
            best_mse = mse
            best_param = param

    mse_val_grids_nn1.append(mse_grid)
    best_param_nn1.append(best_param)
    print(f"\nSplit {split_idx} : meilleurs params NN1 = {best_param} (MSE val = {best_mse:.6f})")

    #concatène des ndarray
    x_trainval = np.vstack([x_train_nn, x_val_nn])
    y_trainval = np.vstack([y_train_nn, y_val_nn])
    
    nn1_final = NeuralNetRegressor(
        Net1,
        verbose=0,
        max_epochs=200,
        optimizer=torch.optim.SGD,
        **nn1_architecture,
        **best_param
    )

    nn1_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = nn1_final.predict(x_trainval)
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_in_sample_list_nn1.append(r2_in)

    # R² out-of-sample
    y_test_pred = nn1_final.predict(x_test_nn)
    r2_out = r2(y_test_nn, y_test_pred)
    r2_test_list_nn1.append(r2_out)

    # Stocker dates, vrais et prédits
    all_dates_nn1.append(dates_test.values)
    all_y_true_nn1.append(y_test_nn)
    all_y_pred_nn1.append(y_test_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² test : {r2_out:.6f}")

# Concaténer résultats test
all_dates_nn1 = np.concatenate(all_dates_nn1)
all_y_true_nn1 = np.concatenate(all_y_true_nn1)
all_y_pred_nn1 = np.concatenate(all_y_pred_nn1)

print([arr.shape for arr in all_y_pred_nn1])

df_results_nn1 = pd.DataFrame({
    "Date": all_dates_nn1,
    "y_true": all_y_true_nn1,
    "NN1": all_y_pred_nn1
})

# Calcul R² mensuel out-of-sample
r2_monthly_all_nn1 = []
for date, group in df_results_nn1.groupby("Date"):
    r2_value = r2(group["y_true"].values, group["NN1"].values)
    r2_monthly_all_nn1.append((date, r2_value))

df_r2_monthly_nn1 = pd.DataFrame(r2_monthly_all_nn1, columns=["Date", "R2_NN1"]).sort_values("Date").reset_index(drop=True)

# Moyennes globales
print("\n🔹 Moyenne R² in-sample (NN1) :", np.nanmean(r2_in_sample_list_nn1))
print("🔹 Moyenne R² oos (NN1) :", np.nanmean(r2_test_list_nn1))
print("🔹 R² mensuel moyen OOS (NN1) :", df_r2_monthly_nn1["R2_NN1"].mean())